# Tariff example usage
This notebook demonstrates how to use the `Tariff` class to calculate energy costs based on a given tariff structure. We will create a sample tariff and then use it to compute the cost of energy consumption over a specified period.

### 1. Creating an index
Most tariffs are based on one or more indexes. For example, a common structure is to have a fixed cost component and a variable cost component that depends on the day-ahead electricity price. Let's create an index for the day-ahead price using the `EntsoeDayAheadIndex`:

In [1]:
from os import environ

from energy_cost.index import EntsoeDayAheadIndex, Index

Index.register("Belpex15min", EntsoeDayAheadIndex(country_code="BE", api_key=environ["ENTSOE_API_KEY"]))

### 2. Defining a tariff
The easiest way to define a tariff is to specify it in a YAML file. A tariff is a list of versions, and each version becomes active from its `start` timestamp.

We'll start with the simplest form: a version with a `consumption` formula.

A simple dynamic tariff might look like this:

In [2]:
from helpers import display_yaml  # ty: ignore[unresolved-import]

display_yaml("../data/tariffs/dynamic_tariff.yml")

```yaml
- start: 2025-01-01T00:00:00+01:00 # the start of validity of this tariff version
  consumption: # the cost formula for consumption
    constant_cost: 10.0 # the constant cost component of the price formula in €/MWh
    variable_costs: # the variable cost components of the price formula in €/MWh
    - index: Belpex15min # the name of the index to use for this variable cost component
      scalar: 1.05 # the scalar to apply to the index values for this variable cost component
- start: 2026-01-01T00:00:00+01:00 # from this date, a new tariff version applies with a different price formula
  consumption:
    constant_cost: 12.0
    variable_costs:
    - index: Belpex15min
      scalar: 1.10
```

In [3]:
from energy_cost.tariff import Tariff

tariff = Tariff.from_yaml("../data/tariffs/dynamic_tariff.yml")
tariff

Tariff(versions=[TariffVersion(start=datetime.datetime(2025, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), injection={}, consumption={'all': {<CostType.ENERGY: 'energy'>: PriceFormula(constant_cost=10.0, variable_costs=[IndexAdder(index='Belpex15min', scalar=1.05)])}}, periodic={}), TariffVersion(start=datetime.datetime(2026, 1, 1, 0, 0, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600))), injection={}, consumption={'all': {<CostType.ENERGY: 'energy'>: PriceFormula(constant_cost=12.0, variable_costs=[IndexAdder(index='Belpex15min', scalar=1.1)])}}, periodic={})])

### 3. Calculating cost per energy consumption in a given period
Once we have defined our tariff, we can use it to calculate the cost of energy consumption over a specified period. This is done by calling the `get_cost` method of the `Tariff` class, which returns a DataFrame with the cost for each time step in the given period.

In [4]:
import datetime as dt

start = dt.datetime.fromisoformat("2026-03-08 00:00:00+01:00")
end = dt.datetime.fromisoformat("2026-03-10 00:00:00+01:00")
tariff.get_cost(start, end)

,timestamp,energy,total
0,2026-03-08 00:00:00+01:00,162.392,162.392
1,2026-03-08 00:15:00+01:00,153.471,153.471
2,2026-03-08 00:30:00+01:00,148.840,148.840
3,2026-03-08 00:45:00+01:00,143.758,143.758
4,2026-03-08 01:00:00+01:00,153.152,153.152
...,...,...,...
187,2026-03-09 22:45:00+01:00,156.144,156.144
188,2026-03-09 23:00:00+01:00,166.374,166.374
189,2026-03-09 23:15:00+01:00,160.665,160.665
190,2026-03-09 23:30:00+01:00,154.472,154.472


### 4. Specifying injection tariffs
The `Tariff` class also supports injection tariffs, which can be used to calculate the revenue from feeding energy back into the grid. To do this, add an `injection` formula next to `consumption` in the tariff version, then call `get_cost` with `direction=PowerDirection.INJECTION`.

In [5]:
display_yaml("../data/tariffs/injection_tariff.yml")

```yaml
# Separate injection and consumption formulas
- start: 2025-01-01T00:00:00+01:00
  injection:
    constant_cost: -14.56
    variable_costs:
    - index: Belpex15min
      scalar: 0.9876
  consumption:
    constant_cost: 12.34
    variable_costs:
    - index: Belpex15min
      scalar: 1.23
```

In [6]:
from energy_cost.tariff import PowerDirection

tariff = Tariff.from_yaml("../data/tariffs/injection_tariff.yml")
tariff.get_cost(start, end, direction=PowerDirection.INJECTION)

,timestamp,energy,total
0,2026-03-08 00:00:00+01:00,120.464672,120.464672
1,2026-03-08 00:15:00+01:00,112.455236,112.455236
2,2026-03-08 00:30:00+01:00,108.297440,108.297440
3,2026-03-08 00:45:00+01:00,103.734728,103.734728
4,2026-03-08 01:00:00+01:00,112.168832,112.168832
...,...,...,...
187,2026-03-09 22:45:00+01:00,114.855104,114.855104
188,2026-03-09 23:00:00+01:00,124.039784,124.039784
189,2026-03-09 23:15:00+01:00,118.914140,118.914140
190,2026-03-09 23:30:00+01:00,113.353952,113.353952


### 5. Specifying meter types
The `Tariff` class also supports different prices per meter type, such as `single_rate`, `tou_peak`, and `tou_offpeak`. When a tariff depends on meter type, you can place those formulas under `consumption` or `injection`.

If needed, you can also add an `all` entry for formulas that apply to every meter type, and override them with a more specific meter type.

In [7]:
display_yaml("../data/tariffs/metered_tariff.yml")

```yaml
# Injection shared across all meter types; consumption varies by meter type
- start: 2025-01-01T00:00:00+01:00
  injection:
    constant_cost: -14.56
    variable_costs:
    - index: Belpex15min
      scalar: 0.9876
  consumption:
    tou_peak:
      constant_cost: 23.45
      variable_costs:
      - index: Belpex15min
        scalar: 2.34
    tou_offpeak:
      constant_cost: 10.0
      variable_costs:
      - index: Belpex15min
        scalar: 0.56

```

In [8]:
from energy_cost.tariff import MeterType

tariff = Tariff.from_yaml("../data/tariffs/metered_tariff.yml")
tariff.get_cost(start, end, meter_type=MeterType.TOU_PEAK)

,timestamp,energy,total
0,2026-03-08 00:00:00+01:00,343.3748,343.3748
1,2026-03-08 00:15:00+01:00,324.3974,324.3974
2,2026-03-08 00:30:00+01:00,314.5460,314.5460
3,2026-03-08 00:45:00+01:00,303.7352,303.7352
4,2026-03-08 01:00:00+01:00,323.7188,323.7188
...,...,...,...
187,2026-03-09 22:45:00+01:00,330.0836,330.0836
188,2026-03-09 23:00:00+01:00,351.8456,351.8456
189,2026-03-09 23:15:00+01:00,339.7010,339.7010
190,2026-03-09 23:30:00+01:00,326.5268,326.5268


### 6. Complex cost components
Some tariffs split the consumption price into separate components. In the example below, `consumption` includes `energy`, `renewable_certificates`, and `chp_certificates`.

This allows `get_cost` to return one column per component, together with the combined `total`.

In [9]:
display_yaml("../data/tariffs/cost_types_tariff.yml")

```yaml
# Multiple cost types in consumption
- start: 2025-01-01T00:00:00+01:00
  injection:
    constant_cost: -14.56
    variable_costs:
    - index: Belpex15min
      scalar: 0.9876
  consumption:
    renewable_certificates:
      constant_cost: 12.0
    chp_certificates:
      constant_cost: 4.56
    energy:
      constant_cost: 12.34
      variable_costs:
      - index: Belpex15min
        scalar: 1.23
```

In [10]:
tariff = Tariff.from_yaml("../data/tariffs/cost_types_tariff.yml")
tariff.get_cost(start, end)

,timestamp,renewable_certificates,chp_certificates,energy,total
0,2026-03-08 00:00:00+01:00,12.0,4.56,180.5056,197.0656
1,2026-03-08 00:15:00+01:00,12.0,4.56,170.5303,187.0903
2,2026-03-08 00:30:00+01:00,12.0,4.56,165.3520,181.9120
3,2026-03-08 00:45:00+01:00,12.0,4.56,159.6694,176.2294
4,2026-03-08 01:00:00+01:00,12.0,4.56,170.1736,186.7336
...,...,...,...,...,...
187,2026-03-09 22:45:00+01:00,12.0,4.56,173.5192,190.0792
188,2026-03-09 23:00:00+01:00,12.0,4.56,184.9582,201.5182
189,2026-03-09 23:15:00+01:00,12.0,4.56,178.5745,195.1345
190,2026-03-09 23:30:00+01:00,12.0,4.56,171.6496,188.2096


### 7. Periodic costs
Some tariffs also include fixed costs that do not depend on energy usage, such as a daily connection fee or a monthly service fee.

These are defined under `periodic`. Use `get_periodic_cost` to calculate the prorated cost over a given time interval:


In [11]:
display_yaml("../data/tariffs/periodic_tariff.yml")

```yaml
# Fixed periodic costs
- start: 2025-01-01T00:00:00+01:00
  periodic:
    connection_fee:
      period: daily
      constant_cost: 1.5
    service_fee:
      period: monthly
      constant_cost: 31.0

```

In [12]:
periodic_tariff = Tariff.from_yaml("../data/tariffs/periodic_tariff.yml")
periodic_tariff.get_periodic_cost(start, end)

{'connection_fee': 3.0, 'service_fee': 2.0}

### 8. Scheduled (time-of-use) price formulas

Some tariffs apply different prices depending on the **day of the week** and/or **time of day**, for example a weekday morning peak and a cheaper fallback during the remaining hours.

For this kind of tariff, `consumption` is written as a list of formulas with optional `when` conditions. The formulas are checked in order, the first match wins, and a formula without `when` acts as a fallback:


In [13]:
display_yaml("../data/tariffs/scheduled_tariff.yml")

```yaml
- start: 2025-01-01T00:00:00+01:00
  consumption:
    - when: # formula valid for weekday mornings and weekend daytime
        - days: [monday, tuesday, wednesday, thursday, friday]
          start: "06:00:00"
          end: "10:00:00"
        - days: [saturday, sunday]
          start: "07:00:00"
          end: "19:00:00"
      constant_cost: 300.0
    - when: # formula valid any day during late morning and evening (the first matching formula takes precedence, so this will not apply during the weekday morning and weekend daytime windows above)
        - start: "10:00:00"
          end: "13:00:00"
        - start: "18:00:00"
          end: "22:00:00"
      constant_cost: 150.0
    - constant_cost: 100.0 # formula valid at all times not covered by the previous definitions

```

In [14]:
import datetime as dt

scheduled_tariff = Tariff.from_yaml("../data/tariffs/scheduled_tariff.yml")

# One full weekday at hourly resolution — Monday 2026-03-16
weekday_start = dt.datetime.fromisoformat("2026-03-16 00:00:00+01:00")
weekday_end = dt.datetime.fromisoformat("2026-03-17 00:00:00+01:00")

scheduled_tariff.get_cost(weekday_start, weekday_end, resolution=dt.timedelta(hours=1))

,timestamp,energy,total
0,2026-03-16 00:00:00+01:00,100.0,100.0
1,2026-03-16 01:00:00+01:00,100.0,100.0
2,2026-03-16 02:00:00+01:00,100.0,100.0
3,2026-03-16 03:00:00+01:00,100.0,100.0
4,2026-03-16 04:00:00+01:00,100.0,100.0
5,2026-03-16 05:00:00+01:00,100.0,100.0
6,2026-03-16 06:00:00+01:00,300.0,300.0
7,2026-03-16 07:00:00+01:00,300.0,300.0
8,2026-03-16 08:00:00+01:00,300.0,300.0
9,2026-03-16 09:00:00+01:00,300.0,300.0


And the same tariff over a full weekend day:


In [15]:
# One full weekend day at hourly resolution — Saturday 2026-03-21
weekend_start = dt.datetime.fromisoformat("2026-03-21 00:00:00+01:00")
weekend_end = dt.datetime.fromisoformat("2026-03-22 00:00:00+01:00")

scheduled_tariff.get_cost(weekend_start, weekend_end, resolution=dt.timedelta(hours=1))

,timestamp,energy,total
0,2026-03-21 00:00:00+01:00,100.0,100.0
1,2026-03-21 01:00:00+01:00,100.0,100.0
2,2026-03-21 02:00:00+01:00,100.0,100.0
3,2026-03-21 03:00:00+01:00,100.0,100.0
4,2026-03-21 04:00:00+01:00,100.0,100.0
5,2026-03-21 05:00:00+01:00,100.0,100.0
6,2026-03-21 06:00:00+01:00,100.0,100.0
7,2026-03-21 07:00:00+01:00,300.0,300.0
8,2026-03-21 08:00:00+01:00,300.0,300.0
9,2026-03-21 09:00:00+01:00,300.0,300.0
